In [41]:
import torch
import torch.nn as nn
from torchinfo import summary
from sklearn.model_selection import train_test_split

In [42]:
data = torch.rand(10,5)

In [43]:
X = data[:,0:-1]
y = torch.tensor([1,0,1,0,1,0,1,0,1,0], dtype=torch.float32) #data[:,-1]

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [45]:
# 4 inputs ---> hidden layer (1 neuron, 4 weights, 1 bias) ---> output layer (1 neuron, 1 weight, 1 bias)
# total parameters = 4*1 + 1 + 1*1 + 1 = 7 parameters

In [46]:
class BasicANN(nn.Module):
    def __init__(self,num_of_features):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=num_of_features,out_features=1)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(in_features=1,out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.sigmoid(x)
        return x
    
    

In [ ]:
#parameters
learning_rate = 0.01
epochs = 10

#model
model = BasicANN(num_of_features=X.shape[1])

#loss function
loss_fn = nn.BCELoss()



In [48]:
summary(model,input_size=(10,4))

Layer (type:depth-idx)                   Output Shape              Param #
BasicANN                                 [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   5
├─ReLU: 1-2                              [10, 1]                   --
├─Linear: 1-3                            [10, 1]                   2
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 7
Trainable params: 7
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [49]:
model

BasicANN(
  (layer_1): Linear(in_features=4, out_features=1, bias=True)
  (relu): ReLU()
  (layer_2): Linear(in_features=1, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [ ]:
#forward pass
y_pred = model(X_train)
# loss
loss = loss_fn(y_pred,y_train.view(-1,1))
# backward pass
loss.backward()
#update weights
with torch.no_grad(): # no_grad() is used to prevent tracking of gradients during weight updates
    model.layer_1.weight -= learning_rate * model.layer_1.weight.grad
    model.layer_1.bias -= learning_rate * model.layer_1.bias.grad

# zero gradients after updating weights to prevent accumulation of gradients in the next iteration
# gradients are accumulated by default in PyTorch, so we need to zero them out after each update
model.layer_1.weight.grad.zero_()
model.layer_1.bias.grad.zero_()

tensor([0.])

In [54]:
for epoch in range(epochs):
    #forward pass
    y_pred = model(X_train)
    #loss
    loss = loss_fn(y_pred,y_train.view(-1,1))
    #backward pass
    loss.backward()
    #update weights
    with torch.no_grad():
        model.layer_1.weight -= learning_rate * model.layer_1.weight.grad
        model.layer_1.bias -= learning_rate * model.layer_1.bias.grad
        model.layer_2.weight -= learning_rate * model.layer_2.weight.grad
        model.layer_2.bias -= learning_rate * model.layer_2.bias.grad

    # zero gradients after updating weights
    model.layer_1.weight.grad.zero_()
    model.layer_1.bias.grad.zero_()
    model.layer_2.weight.grad.zero_()
    model.layer_2.bias.grad.zero_()


    print(f"Epoch: {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch: 1/10, Loss: 0.7528
Epoch: 2/10, Loss: 0.7519
Epoch: 3/10, Loss: 0.7517
Epoch: 4/10, Loss: 0.7514
Epoch: 5/10, Loss: 0.7511
Epoch: 6/10, Loss: 0.7508
Epoch: 7/10, Loss: 0.7506
Epoch: 8/10, Loss: 0.7503
Epoch: 9/10, Loss: 0.7500
Epoch: 10/10, Loss: 0.7498


In [ ]:
with torch.no_grad():
  # note: we don't need to calculate gradients during evaluation, so we use no_grad() to save memory and computation
  y_pred = model(X_test)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy : {accuracy}')

Accuracy : 0.5


In [56]:
model.state_dict()

OrderedDict([('layer_1.weight',
              tensor([[ 0.4190, -0.4007, -0.2972,  0.0404]])),
             ('layer_1.bias', tensor([-0.4493])),
             ('layer_2.weight', tensor([[-0.5374]])),
             ('layer_2.bias', tensor([0.6777]))])